# Grid Search — the loop, written once

The previous notebook ended with a nested loop that was already getting unwieldy at two
hyperparameters. `GridSearchCV` is that loop, generalised: give it a model and a dictionary
of values to try, and it fits every combination with cross-validation and remembers the
best.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RSEED = 42

# Predict a penguin's sex from its measurements. Species prediction would be too easy —
# nearly perfectly separable — and a tuning lesson needs a task where the settings matter.
penguins = pd.read_csv("data/penguins.csv").dropna(subset=["sex"])

X = penguins.drop(columns="sex")
y = penguins["sex"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RSEED, stratify=y
)

print(f"{len(X_train)} training penguins, {len(X_test)} held out")
print(y.value_counts().to_dict())

In [ ]:
numeric = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
categorical = make_pipeline(SimpleImputer(strategy="most_frequent"),
                            OneHotEncoder(handle_unknown="ignore"))

preprocessor = make_column_transformer(
    (numeric, make_column_selector(dtype_include="number")),
    (categorical, make_column_selector(dtype_exclude="number")),
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])
model

## The search space

A `param_grid` is a dictionary: keys are the `get_params()` names from the last notebook,
values are the lists to try.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "preprocessor__pipeline-1__simpleimputer__strategy": ["mean", "median"],
}

n_combinations = 5 * 2
print(f"{n_combinations} combinations × 5 folds = {n_combinations * 5} fits")

**Count the fits before you press go.** The cost is the product of every list length,
multiplied by the number of folds. Five values of one parameter and two of another is ten
combinations — but add a third parameter with six values and you are at 300 fits, and a
fourth takes you past 1,000. Grid search grows multiplicatively, and it is easy to launch
something that runs for hours by accident.

In [ ]:
search = GridSearchCV(
    model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,        # use all cores
    return_train_score=True,
)

search.fit(X_train, y_train)

print(f"best parameters : {search.best_params_}")
print(f"best cv score   : {search.best_score_:.4f}")

## What you get back

Three attributes matter:

- **`best_params_`** — the winning combination.
- **`best_score_`** — its mean cross-validated score. *Not* an estimate of test performance,
  for the reason the last notebook laid out.
- **`best_estimator_`** — the model refitted on the **whole** training set using those
  parameters. This is the one to use.

That refit happens automatically (`refit=True` is the default), which is why the search
object itself can predict.

In [ ]:
print(type(search.best_estimator_).__name__)

# the search behaves as the fitted best model
print(f"test accuracy: {search.score(X_test, y_test):.4f}")

## `cv_results_` — the leaderboard

`best_params_` tells you the winner and nothing else. The full results tell you *whether the
winner mattered*, which is usually the more useful question.

In [ ]:
results = pd.DataFrame(search.cv_results_)

leaderboard = results[[
    "param_classifier__C",
    "param_preprocessor__pipeline-1__simpleimputer__strategy",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]].sort_values("rank_test_score")

leaderboard.columns = ["C", "imputer", "train", "cv score", "std", "rank"]
leaderboard.round(4)

Read the `std` column next to `cv score`. If the top several rows differ by less than their
standard deviations, **the ranking is noise** — you have learned that these settings do not
much matter, which is a real and useful finding. Reporting "we tuned it and got the best
model" would be overclaiming.

Read `train` against `cv score` too. A combination where training score is far above the
cross-validated score is overfitting, even if it happens to rank well.

In [ ]:
# how much does the choice actually matter?
spread = results["mean_test_score"].max() - results["mean_test_score"].min()
typical_noise = results["std_test_score"].mean()

print(f"best − worst across the grid : {spread:.4f}")
print(f"typical fold-to-fold std     : {typical_noise:.4f}")
print()
print("the search is meaningful" if spread > 2 * typical_noise
      else "the differences are within noise — these settings barely matter here")

## Seeing the whole search at once

With two hyperparameters a pivot table is readable. With four or five it is not, and a
**parallel-coordinate plot** is the standard way to look at a high-dimensional search: one
vertical axis per hyperparameter plus one for the score, and one line per combination.

Lines that end high on the score axis are good combinations; where they cluster on the other
axes tells you which regions of the space were worth being in.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

pivot = results.pivot_table(
    index="param_classifier__C",
    columns="param_preprocessor__pipeline-1__simpleimputer__strategy",
    values="mean_test_score",
)

for strategy in pivot.columns:
    ax.plot(pivot.index, pivot[strategy], marker="o", label=f"imputer = {strategy}")

ax.set_xscale("log")
ax.set_xlabel("C (log scale)")
ax.set_ylabel("cross-validated accuracy")
ax.set_title("Every combination in the grid")
ax.legend()
plt.show()

The shape here is the useful part: accuracy climbs steeply as `C` rises out of the
over-regularised region and then flattens. The imputation strategy barely separates the two
lines at all.

**A flat region means the parameter does not matter in that range.** That is worth knowing —
it tells you where *not* to spend your next search.

## Summary

- `GridSearchCV(model, param_grid, cv=…)` tries every combination and refits the winner.
- Cost is **multiplicative**: count `combinations × folds` before launching.
- `best_params_`, `best_score_`, `best_estimator_` — and `best_score_` is not a performance
  estimate.
- `cv_results_` is the real output. Compare the spread of scores against their standard
  deviation before believing a ranking.

> **Next:** exhaustive search wastes most of its effort on regions that do not matter. Random
> search does better for the same budget.